# Detection Scorers & Empirical Threshold Simulation

Demonstrates:
- Unified `score_profiles` across `boxcar_glrt` (heteroscedastic GLRT handling missing/zero-weight bins), `double_boxcar` (disjoint two-window interpulse scorer), `double_matched`, `combined` (boxcar + harmonic), and `kadane`.
- Empirical non-Gaussian threshold calibration with `threshold_simulation`.

In [1]:
import numpy as np

from pyloki.detection import scoring, thresholding

rng = np.random.default_rng(7)
n_profiles, n_bins = 4, 64
widths = scoring.generate_box_width_trials(n_bins, ducy_max=0.25, wtsp=1.5)

# Heteroscedastic (E, V) profiles with a double-peaked interpulse in profile 0
ts_v = rng.uniform(0.5, 2.0, size=(n_profiles, n_bins)).astype(np.float32)
ts_e = (rng.normal(0.0, 1.0, size=(n_profiles, n_bins)) * np.sqrt(ts_v)).astype(
    np.float32
)
ts_e[0, 16:20] += 3.5 * np.sqrt(ts_v[0, 16:20])  # main pulse
ts_e[0, 46:50] += 3.0 * np.sqrt(ts_v[0, 46:50])  # interpulse

ev = np.stack([ts_e, ts_v], axis=1)
for mode in ("boxcar_glrt", "double_boxcar", "double_matched", "combined", "kadane"):
    scores = scoring.score_profiles(ev, widths, mode=mode)
    print(f"mode={mode:14s} -> profile 0 S/N = {float(scores[0]):6.2f}")

# GLRT also safely handles missing / zero-weight phase bins
ev_gap = ev.copy()
ev_gap[:, :, 0:4] = 0.0
glrt_gap = scoring.score_profiles(ev_gap, widths, mode="boxcar_glrt")
print(
    f"boxcar_glrt with 4 zero-weight bins -> profile 0 S/N = {float(glrt_gap[0]):6.2f}"
)

mode=boxcar_glrt    -> profile 0 S/N =   6.18
mode=double_boxcar  -> profile 0 S/N =   6.29
mode=double_matched -> profile 0 S/N =   6.24
mode=combined       -> profile 0 S/N =   6.18
mode=kadane         -> profile 0 S/N =   6.18
boxcar_glrt with 4 zero-weight bins -> profile 0 S/N =   6.09


In [2]:
# Empirical Monte Carlo threshold calibration for an 8-stage EP tree
branching_pattern = np.full(8, 2.0, dtype=np.float64)
survive_probs = np.full(8, 0.5, dtype=np.float64)
thresholds, state = thresholding.threshold_simulation(
    survive_probs,
    branching_pattern,
    ref_ducy=0.05,
    nbins=64,
    ntrials=2000,
    snr_final=9.0,
    score_mode="boxcar_glrt",
    seed=11,
)
print("Simulated thresholds:", np.round(thresholds, 2))
print(
    f"Final H1 survival = {state.entries[-1].success_h1_cumul:.3f}, "
    f"total cost = {state.entries[-1].cost:.2f}"
)

Simulated thresholds: [2.66 2.85 3.   3.16 3.34 3.51 3.63 3.79]
Final H1 survival = 0.976, total cost = 17.42
